**Keystroke Dynamics for Continuous Authentication**

A behavioral biometric security system that uses an LSTM to verify a user's identity from their typing rhythm.

**Dataset**: CMU Keystroke Dynamics Benchmark Dataset (Killourhy & Maynard) — 51 subjects, each typing the password '.tie5Roanl' 400 times across 8 sessions. Each row provides pre-extracted timing features per keystroke: Hold time (H), Down-Down (DD), and Up-Down/flight time (UD).

**DATA LOADING AND EXPLORATION**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
df=pd.read_csv('/content/DSL-StrongPasswordData.csv')

In [4]:
df.head()

,subject,sessionIndex,rep,H.period,DD.period.t,UD.period.t,H.t,DD.t.i,UD.t.i,H.i,...,H.a,DD.a.n,UD.a.n,H.n,DD.n.l,UD.n.l,H.l,DD.l.Return,UD.l.Return,H.Return
0,s002,1,1,0.1491,0.3979,0.2488,0.1069,0.1674,0.0605,0.1169,...,0.1349,0.1484,0.0135,0.0932,0.3515,0.2583,0.1338,0.3509,0.2171,0.0742
1,s002,1,2,0.1111,0.3451,0.2340,0.0694,0.1283,0.0589,0.0908,...,0.1412,0.2558,0.1146,0.1146,0.2642,0.1496,0.0839,0.2756,0.1917,0.0747
2,s002,1,3,0.1328,0.2072,0.0744,0.0731,0.1291,0.0560,0.0821,...,0.1621,0.2332,0.0711,0.1172,0.2705,0.1533,0.1085,0.2847,0.1762,0.0945
3,s002,1,4,0.1291,0.2515,0.1224,0.1059,0.2495,0.1436,0.1040,...,0.1457,0.1629,0.0172,0.0866,0.2341,0.1475,0.0845,0.3232,0.2387,0.0813
4,s002,1,5,0.1249,0.2317,0.1068,0.0895,0.1676,0.0781,0.0903,...,0.1312,0.1582,0.0270,0.0884,0.2517,0.1633,0.0903,0.2517,0.1614,0.0818


In [5]:
df.shape

(20400, 34)

In [34]:
df.isnull().sum()

,0
subject,0
sessionIndex,0
rep,0
H.period,0
DD.period.t,0
UD.period.t,0
H.t,0
DD.t.i,0
UD.t.i,0
H.i,0


In [6]:
df.drop('H.Return',inplace=True,axis=1)

dropped the least informative feature to get exactly 30 features so that it could be reshaped into 30*3 instead of padding since it adds up a fabricated piece of information

**FEATURE ENGINEERING: BUILDING GENUINE AND IMPOSTER LABELS**

In [7]:
def dataset(df,target,imposter_sample=200):
  features=[c for c in df.columns if c not in ['subject','sessionIndex','rep']]

  genuine=df[df['subject']==target]
  imposter=df[df['subject']!=target].sample(imposter_sample)
  y_genuine=np.ones(len(genuine))
  y_imposter=np.zeros(len(imposter))

  genuine=genuine[features].values
  imposter=imposter[features].values
  return genuine, imposter, y_genuine, y_imposter

In [8]:
gen_data, imp_data, y_gen, y_imp = dataset(df, target='s002', imposter_sample=200)
x = np.vstack([gen_data, imp_data])
y = np.concatenate([y_gen, y_imp])

In [9]:
x.shape

(600, 30)

In [10]:
y.shape

(600,)

**RESHAPING FOR THE LSTM**

In [11]:
def for_lstm(x):
  return x.reshape(x.shape[0], 10, 3)

In [12]:
x_seq=for_lstm(x)

In [13]:
x_seq.shape

(600, 10, 3)

**TRAIN TEST SPLIT AND SCALING**

In [14]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [15]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [16]:
x_train_seq=for_lstm(x_train)
x_test_seq=for_lstm(x_test)

In [17]:
x_train_seq.shape

(480, 10, 3)

In [18]:
x_test_seq.shape

(120, 10, 3)

**TENSOR CONVERSION AND DATALOADERS**

In [19]:
x_train_seq=torch.from_numpy(x_train_seq).float()
x_test_seq=torch.from_numpy(x_test_seq).float()
y_train=torch.from_numpy(y_train).float()
y_test=torch.from_numpy(y_test).float()

In [20]:
y_train=y_train.unsqueeze(1)
y_test=y_test.unsqueeze(1)

In [21]:
print(y_train.shape,y_test.shape)

torch.Size([480, 1]) torch.Size([120, 1])


In [22]:
train_dataset=TensorDataset(x_train_seq,y_train)
test_dataset=TensorDataset(x_test_seq,y_test)

In [23]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [24]:
for x_batch, y_batch in train_loader:
    print(x_batch.shape, y_batch.shape)
    break

torch.Size([32, 10, 3]) torch.Size([32, 1])


**MODEL ARCHITECTURE**

In [25]:
class keystroke(nn.Module):
  def __init__(self,input_size,hidden_size):
    super().__init__()
    self.lstm=nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
    self.dropout=nn.Dropout(0.2)
    self.fc=nn.Linear(in_features=hidden_size, out_features=1)

  def forward(self, x):
    lstm_out, (h_n, c_n) = self.lstm(x)
    last_hidden = h_n[-1]
    dropped = self.dropout(last_hidden)
    logits = self.fc(dropped)
    out = torch.sigmoid(logits)
    return out

In [26]:
model = keystroke(input_size=3, hidden_size=32)
print(model)

keystroke(
  (lstm): LSTM(3, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [27]:
sample_out = model(x_batch)
print(sample_out.shape)

torch.Size([32, 1])


**TRAINING LOOP**

In [28]:
criterion=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [29]:
epochs=50
for epoch in range(epochs):
  model.train()
  total_loss=0

  for x_batch,y_batch in train_loader:
    optimizer.zero_grad()
    outputs=model(x_batch)
    loss=criterion(outputs,y_batch)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()

  avg_loss=total_loss/len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/50, Loss: 0.7072
Epoch 2/50, Loss: 0.6749
Epoch 3/50, Loss: 0.6318
Epoch 4/50, Loss: 0.6026
Epoch 5/50, Loss: 0.5953
Epoch 6/50, Loss: 0.5769
Epoch 7/50, Loss: 0.5617
Epoch 8/50, Loss: 0.5429
Epoch 9/50, Loss: 0.5281
Epoch 10/50, Loss: 0.5012
Epoch 11/50, Loss: 0.4830
Epoch 12/50, Loss: 0.4702
Epoch 13/50, Loss: 0.4507
Epoch 14/50, Loss: 0.4324
Epoch 15/50, Loss: 0.4099
Epoch 16/50, Loss: 0.3986
Epoch 17/50, Loss: 0.3807
Epoch 18/50, Loss: 0.3680
Epoch 19/50, Loss: 0.3484
Epoch 20/50, Loss: 0.3411
Epoch 21/50, Loss: 0.3245
Epoch 22/50, Loss: 0.3375
Epoch 23/50, Loss: 0.3077
Epoch 24/50, Loss: 0.2898
Epoch 25/50, Loss: 0.2874
Epoch 26/50, Loss: 0.2773
Epoch 27/50, Loss: 0.2670
Epoch 28/50, Loss: 0.2784
Epoch 29/50, Loss: 0.2544
Epoch 30/50, Loss: 0.2393
Epoch 31/50, Loss: 0.2281
Epoch 32/50, Loss: 0.2460
Epoch 33/50, Loss: 0.2295
Epoch 34/50, Loss: 0.2309
Epoch 35/50, Loss: 0.2220
Epoch 36/50, Loss: 0.2167
Epoch 37/50, Loss: 0.2323
Epoch 38/50, Loss: 0.2207
Epoch 39/50, Loss: 0.

**EVALUATION AND FINAL RESULTS**

In [30]:
model.eval()

keystroke(
  (lstm): LSTM(3, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

In [31]:
all_probs = []
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        outputs = model(x_batch)
        all_probs.append(outputs)
        all_labels.append(y_batch)

all_probs = torch.cat(all_probs).numpy()
all_labels = torch.cat(all_labels).numpy()

**THRESHOLD TUNING**

In [32]:
for threshold in [0.3, 0.5, 0.7, 0.9]:
    preds = (all_probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
    far = fp / (fp + tn)
    frr = fn / (fn + tp)
    print(f"Threshold {threshold}: FAR={far:.2%}, FRR={frr:.2%}")

Threshold 0.3: FAR=27.50%, FRR=6.25%
Threshold 0.5: FAR=20.00%, FRR=12.50%
Threshold 0.7: FAR=12.50%, FRR=20.00%
Threshold 0.9: FAR=10.00%, FRR=31.25%


In [33]:
preds_07 = (all_probs > 0.7).astype(int)
print(classification_report(all_labels, preds_07, target_names=['Imposter', 'Genuine']))
print(confusion_matrix(all_labels, preds_07))

              precision    recall  f1-score   support

    Imposter       0.69      0.88      0.77        40
     Genuine       0.93      0.80      0.86        80

    accuracy                           0.82       120
   macro avg       0.81      0.84      0.81       120
weighted avg       0.85      0.82      0.83       120

[[35  5]
 [16 64]]
